# Comparación de Random Forest y Extra Trees en ASSISTments

Notebook complementario de la práctica 1-2. Compara tres corridas sobre el mismo CSV procesado de 3 000 estudiantes y la misma partición oficial: Random Forest de la práctica, Random Forest con hiperparámetros modificados y Extra Trees, algoritmo incluido en el estudio original [1].

La comparación conserva las 722 variables, las etiquetas score y la división source_split. Los parámetros de la segunda corrida se fijan manualmente; no se ejecuta búsqueda automática. Las métricas de entrenamiento describen ajuste y las de evaluation se calculan sobre los registros reservados.


## 1. Carga del CSV y revisión de datos

La preparación de variables se realiza en ASSISTments_01_Preparar_CSV.ipynb. Este notebook consume su salida assistments_rf_dataset.csv.gz; no vuelve a procesar los logs crudos. El archivo de referencia tiene 3 000 estudiantes, 51 042 filas, 727 columnas y 722 predictores.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# En Colab, sube el CSV procesado si no está en la ruta configurada.
# En Jupyter local se usa la ruta del proyecto.
DATASET_CSV = Path('EDMCup2023_fuente_oficial/salida/assistments_rf_dataset.csv.gz')
if not DATASET_CSV.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Sube exactamente un archivo CSV comprimido.')
        uploaded_name = next(iter(uploaded))
        DATASET_CSV = Path('/content') / uploaded_name
    except ImportError as exc:
        raise FileNotFoundError(
            'No se encontró el CSV. Coloca assistments_rf_dataset.csv.gz en la ruta del proyecto.'
        ) from exc

df = pd.read_csv(DATASET_CSV, low_memory=False)
required = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltan columnas requeridas: {sorted(missing)}')

print(f'Archivo: {DATASET_CSV.name}')
print(f'Filas: {len(df):,} | estudiantes: {df.student_id.nunique():,} | columnas: {df.shape[1]:,}')
display(df['source_split'].value_counts(dropna=False).rename_axis('partición').to_frame('filas'))
display(df['score'].value_counts(dropna=False).sort_index().rename_axis('score').to_frame('filas'))


## 2. Variable objetivo, predictores y partición

y es la variable dependiente y se obtiene de score. La clase 0 representa no éxito y la clase 1 éxito, de acuerdo con la etiqueta oficial. X contiene las variables independientes procesadas. Se excluyen score, los identificadores y source_split para evitar que la etiqueta o la partición entren como predictores. Solo se conservan filas etiquetadas con 0 o 1.

Cuando el CSV incluye las dos particiones oficiales, se conserva train y evaluation. Si falta una, se crea una validación interna agrupada por estudiante sobre train.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Etiqueta, IDs y partición se conservan en df, pero no son predictores.
id_columns = {
    'score', 'student_id', 'source_split',
    'unit_test_assignment_log_id', 'problem_id'
}
feature_columns = [c for c in df.columns if c not in id_columns]
if not feature_columns:
    raise ValueError('El CSV no contiene predictores.')

# Los predictores ya vienen procesados; aquí se asegura el tipo numérico.
X = df[feature_columns].apply(pd.to_numeric, errors='coerce').fillna(-1).astype('float32')
y = pd.to_numeric(df['score'], errors='coerce')
valid = y.notna() & y.isin([0, 1])
df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype('int8').reset_index(drop=True)

train_mask = df['source_split'].eq('train')
eval_mask = df['source_split'].eq('evaluation')
if train_mask.any() and eval_mask.any():
    # Se usa la partición oficial; no se vuelve a sortear la evaluación.
    train_idx = np.flatnonzero(train_mask.to_numpy())
    test_idx = np.flatnonzero(eval_mask.to_numpy())
    evaluation_name = 'Partición oficial evaluation'
else:
    # Respaldo: los registros de un mismo estudiante no se dividen entre train/test.
    train_candidates = np.flatnonzero(train_mask.to_numpy())
    groups = df.loc[train_candidates, 'student_id'].astype('string').to_numpy()
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    relative_train, relative_test = next(
        splitter.split(X.iloc[train_candidates], y.iloc[train_candidates], groups=groups)
    )
    train_idx = train_candidates[relative_train]
    test_idx = train_candidates[relative_test]
    evaluation_name = 'Validación interna agrupada por estudiante (80/20)'

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
print(f'Evaluación: {evaluation_name}')
print(f'Predictores: {len(feature_columns)} | train: {len(train_idx):,} | evaluación: {len(test_idx):,}')
train_students = set(df.iloc[train_idx]['student_id'].dropna())
eval_students = set(df.iloc[test_idx]['student_id'].dropna())
overlap = train_students & eval_students
shared_eval_rows = df.iloc[test_idx]['student_id'].isin(train_students).sum()
print(f'Estudiantes compartidos entre particiones: {len(overlap)}')
print(f'Registros de evaluación de estudiantes compartidos: {shared_eval_rows:,}')


## 3. Modelos e hiperparámetros

Random Forest práctica reproduce la configuración usada en la práctica 1-2: 600 árboles, profundidad máxima 19, mínimo 10 observaciones para dividir, mínimo 2 por hoja, max_features=sqrt, bootstrap y semilla 42.

Random Forest modificado conserva el algoritmo y cambia profundidad a 12, min_samples_split a 20 y min_samples_leaf a 5. Son parámetros manuales para comparar árboles más restringidos; no se presentan como óptimos.

Extra Trees se compara en el artículo original [1]. Usa la misma muestra, predictores, partición, cantidad de árboles y límites del Random Forest base. Extra Trees incorpora aleatoriedad en los umbrales de división, además de la selección aleatoria de predictores.

En los tres modelos, n_jobs=-1 utiliza los núcleos disponibles y random_state=42 fija la aleatoriedad. No se realiza búsqueda automática de parámetros ni ajuste del umbral.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

SEED = 42
N_ESTIMATORS = 600

models = {
    'RF práctica': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles de la práctica
        max_depth=19,              # profundidad máxima de la práctica
        min_samples_split=10,      # mínimo de datos para dividir un nodo
        min_samples_leaf=2,        # mínimo de datos en una hoja
        max_features='sqrt',       # candidatos por división: raíz del número de predictores
        bootstrap=True,            # cada árbol usa una muestra con reemplazo
        random_state=SEED,         # hace reproducible el ajuste
        n_jobs=-1                  # usa todos los núcleos disponibles
    ),
    'RF parámetros modificados': RandomForestClassifier(
        n_estimators=N_ESTIMATORS, # cantidad de árboles controlada
        max_depth=12,              # limita la complejidad de cada árbol
        min_samples_split=20,      # exige más datos antes de dividir
        min_samples_leaf=5,        # evita hojas con muy pocos registros
        max_features='sqrt',       # conserva la selección de predictores
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    ),
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=19,
        min_samples_split=10,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,            # iguala el muestreo con el RF base
        random_state=SEED,
        n_jobs=-1
    )
}

parameter_table = pd.DataFrame({name: model.get_params() for name, model in models.items()})
display(parameter_table.loc[
    ['n_estimators','max_depth','min_samples_split','min_samples_leaf',
     'max_features','bootstrap','random_state','n_jobs']
].T)


## 4. Entrenamiento y métricas

Cada modelo se ajusta una sola vez con `X_train` e `y_train`. Después se calcula cada métrica en dos conjuntos distintos. **Train** contiene las filas que el algoritmo ya utilizó para aprender: sus resultados describen el ajuste y pueden ser optimistas. **Evaluation** contiene filas que no participaron en `fit`; estima el comportamiento en registros reservados y es la base de la comparación descriptiva.

La tabla informa accuracy, precisión, recall y F1 por clase, sus promedios macro, recall de score=0 y ROC-AUC de score=0. Precisión responde qué proporción de las predicciones de una clase fue correcta; recall, qué proporción de los casos reales de esa clase se encontró; F1 combina ambas. Macro calcula cada clase por separado y promedia con el mismo peso. ROC-AUC se calcula con probabilidades, no con la clase ya asignada, y resume la separación al recorrer umbrales. El recall de score=0 se destaca porque indica cuántos no éxitos se detectan.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)

metric_rows = []
roc_data = {}
confusion = {}

for model_name, model in models.items():
    # El modelo aprende solamente de la partición de entrenamiento.
    model.fit(X_train, y_train)
    roc_data[model_name] = {}
    confusion[model_name] = {}

    for split_name, X_part, y_part in [
        ('train', X_train, y_train),
        ('evaluation', X_test, y_test)
    ]:
        predicted = model.predict(X_part)
        probabilities = model.predict_proba(X_part)
        class_index = {int(label): i for i, label in enumerate(model.classes_)}
        p0 = probabilities[:, class_index[0]]
        y0 = (y_part == 0).astype('int8')

        cm = confusion_matrix(y_part, predicted, labels=[0, 1])
        confusion[model_name][split_name] = cm
        fpr, tpr, thresholds = roc_curve(y0, p0)
        auc_value = roc_auc_score(y0, p0) if len(np.unique(y0)) == 2 else np.nan
        roc_data[model_name][split_name] = (fpr, tpr, auc_value)

        metric_rows.append({
            'Algoritmo': model_name,
            'Partición': split_name,
            'Registros': len(y_part),
            'Accuracy': accuracy_score(y_part, predicted),
            'Precision score=0': precision_score(y_part, predicted, pos_label=0, zero_division=0),
            'Recall score=0': recall_score(y_part, predicted, pos_label=0, zero_division=0),
            'F1 score=0': f1_score(y_part, predicted, pos_label=0, zero_division=0),
            'Precision score=1': precision_score(y_part, predicted, pos_label=1, zero_division=0),
            'Recall score=1': recall_score(y_part, predicted, pos_label=1, zero_division=0),
            'F1 score=1': f1_score(y_part, predicted, pos_label=1, zero_division=0),
            'Precision macro': precision_score(y_part, predicted, average='macro', zero_division=0),
            'Recall macro': recall_score(y_part, predicted, average='macro', zero_division=0),
            'F1 macro': f1_score(y_part, predicted, average='macro', zero_division=0),
            'ROC-AUC score=0': auc_value
        })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(4))
metrics_df.to_csv('metricas_comparacion_RF_ExtraTrees.csv', index=False)
print('Guardado: metricas_comparacion_RF_ExtraTrees.csv')


## 5. Cómo leer las métricas de entrenamiento y evaluación

**Train (39 935 filas):** una accuracy de 0,8347 significa que el RF de la práctica clasificó correctamente el 83,47 % de las mismas filas usadas para ajustar sus árboles. El recall de score=0 de 0,7478 indica que encontró el 74,78 % de los no éxitos presentes en train. Estas cifras describen ajuste; no deben presentarse como rendimiento independiente.

**Evaluation (11 107 filas):** la accuracy de 0,7022 corresponde a aciertos en registros que no entraron en `fit`. El recall de score=0 de 0,5570 significa que detectó el 55,70 % de los no éxitos reales. La brecha train−evaluation permite describir cuánto bajan las métricas fuera del ajuste; una brecha grande es compatible con sobreajuste, pero por sí sola no lo demuestra.

En la matriz de confusión, las filas son la clase real y las columnas la predicha. Para score=0, 2 526 de 4 535 casos fueron detectados y 2 009 quedaron como score=1. La precisión de score=0 fue 0,6604: 2 526 de las 3 825 predicciones de alerta correspondieron a no éxitos; 1 299 fueron score=1. El F1 macro de 0,6828 promedia F1 score=0 (0,6043) y F1 score=1 (0,7612). El AUC=0,7528 mide ordenamiento por probabilidad para distintos umbrales; no significa 75,28 % de predicciones correctas.

La partición oficial separa registros, pero 31 estudiantes aparecen en ambos conjuntos (552 filas de evaluation). Además, evaluation se usa para comparar las tres configuraciones. Por ello estos resultados no validan el desempeño en estudiantes completamente nuevos; la confirmación final debe hacerse en una partición agrupada por estudiante que no se use para elegir modelos ni umbrales. En esta corrida no se ajustó el umbral.

In [ ]:
# Brechas de train menos evaluation: positivas grandes requieren revisar sobreajuste.
gap_rows = []
for model_name in models:
    train_row = metrics_df[(metrics_df['Algoritmo'] == model_name) & (metrics_df['Partición'] == 'train')].iloc[0]
    eval_row = metrics_df[(metrics_df['Algoritmo'] == model_name) & (metrics_df['Partición'] == 'evaluation')].iloc[0]
    for metric in ['Accuracy','F1 macro','Recall score=0','ROC-AUC score=0']:
        gap_rows.append({
            'Algoritmo': model_name,
            'Métrica': metric,
            'Train': train_row[metric],
            'Evaluation': eval_row[metric],
            'Brecha train - evaluation': train_row[metric] - eval_row[metric]
        })
gap_df = pd.DataFrame(gap_rows)
display(gap_df.round(4))


### Lectura numérica de la línea base

La celda siguiente obtiene los conteos directamente de la matriz de confusión de evaluation. Sirve para explicar qué significan recall y precisión de la clase no exitosa y por qué las métricas de train no sustituyen a las de evaluación.

In [ ]:
base_train = metrics_df[(metrics_df['Algoritmo'] == 'RF práctica') & (metrics_df['Partición'] == 'train')].iloc[0]
base_eval = metrics_df[(metrics_df['Algoritmo'] == 'RF práctica') & (metrics_df['Partición'] == 'evaluation')].iloc[0]
cm = confusion['RF práctica']['evaluation']  # filas: real 0/1; columnas: predicho 0/1

no_exitos_reales = int(cm[0, 0] + cm[0, 1])
alertas_score0 = int(cm[0, 0] + cm[1, 0])
print(f"TRAIN ({int(base_train['Registros']):,} filas usadas en fit): accuracy={base_train['Accuracy']:.4f}; recall score=0={base_train['Recall score=0']:.4f}. Describe el ajuste a datos vistos.")
print(f"EVALUATION ({int(base_eval['Registros']):,} filas reservadas): accuracy={base_eval['Accuracy']:.4f} = {int(cm.trace()):,}/{int(base_eval['Registros']):,} aciertos.")
print(f"Score=0: {int(cm[0, 0]):,}/{no_exitos_reales:,} detectados; {int(cm[0, 1]):,} no éxitos quedaron como score=1. Recall={base_eval['Recall score=0']:.4f}.")
print(f"Alertas predichas score=0: {int(cm[0, 0]):,}/{alertas_score0:,} fueron no éxitos; {int(cm[1, 0]):,} fueron éxitos. Precisión={base_eval['Precision score=0']:.4f}.")
print(f"F1 macro={base_eval['F1 macro']:.4f}; AUC score=0={base_eval['ROC-AUC score=0']:.4f}. AUC evalúa probabilidades al variar el umbral, no la fracción de aciertos.")

## 6. Curvas ROC de entrenamiento y evaluación

Cada panel usa como positivo score=0 y la probabilidad estimada para esa clase. El eje horizontal es la tasa de falsos positivos y el vertical la tasa de verdaderos positivos. El AUC aparece en la leyenda. La curva de train suele ser más alta porque el modelo aprendió esos registros; se compara con evaluation para observar cuánto baja la separación en datos reservados.


In [ ]:
colors = ['#1f4e79', '#d28e00', '#24805a']
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=180)
for ax, split_name, title in zip(axes, ['train','evaluation'], ['Entrenamiento','Evaluación']):
    for (model_name, _), color in zip(models.items(), colors):
        fpr, tpr, auc_value = roc_data[model_name][split_name]
        ax.plot(fpr, tpr, lw=1.7, color=color,
                label=f'{model_name} (AUC={auc_value:.3f})')
    ax.plot([0,1], [0,1], '--', color='gray', lw=1)
    ax.set(xlim=(0,1), ylim=(0,1.02),
           xlabel='Tasa de falsos positivos',
           ylabel='Tasa de verdaderos positivos', title=title)
    ax.grid(alpha=.2)
    ax.legend(fontsize=7, loc='lower right')
fig.suptitle('Curvas ROC para la clase no exitosa (score=0)')
fig.tight_layout()
fig.savefig('curvas_ROC_train_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: curvas_ROC_train_evaluation.png')


## 7. Matrices de confusión y limitaciones

La matriz se ordena con filas = clase real y columnas = clase predicha. Los valores de la diagonal son aciertos. La celda real 0/predicho 1 cuenta casos no exitosos que el modelo clasificó como exitosos. El valor real 1/predicho 0 cuenta éxitos que el modelo marcó como no exitosos.

La partición oficial contiene 31 estudiantes compartidos entre train y evaluation (552 registros de evaluación). La comparación reproduce la partición del conjunto disponible, pero no demuestra rendimiento en estudiantes totalmente no vistos. Además, la configuración modificada es una comparación manual y no una búsqueda optimizada.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), dpi=180)
for ax, (model_name, _) in zip(axes, models.items()):
    cm = confusion[model_name]['evaluation']
    ax.imshow(cm, cmap='Blues')
    ax.set_title(model_name, fontsize=9)
    ax.set_xticks([0,1], ['Predicho 0','Predicho 1'], fontsize=7)
    ax.set_yticks([0,1], ['Real 0','Real 1'], fontsize=7)
    for (row, col), value in np.ndenumerate(cm):
        ax.text(col, row, f'{value:,}', ha='center', va='center', fontsize=8,
                color='white' if value > cm.max() * .55 else 'black')
fig.suptitle('Matrices de confusión en evaluation')
fig.tight_layout()
fig.savefig('matrices_confusion_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print('Guardado: matrices_confusion_evaluation.png')


## Referencia

[1] A. Yaramala, S. Farokhi y H. Karimi, “Navigating the Data-Rich Landscape of Online Learning: Insights and Predictions from ASSISTments,” Proceedings of the 17th International Conference on Educational Data Mining, pp. 321–331, Atlanta, Georgia, EE. UU., jul. 2024.
